# 3.2 Company Characterization

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif,mutual_info_classif, chi2
import joblib

# 1. load data
df=pd.read_csv('train_data.csv')
df.columns = df.columns.str.strip().str.replace(' ', '_')


In [3]:
# Save the target
y = df['Bankrupt?']
# Save index
index = df['Index']
# 2. drop 'Index','Bankrupt?'
X = df.drop(columns=['Index','Bankrupt?'])
print(len(X.columns))

95


In [4]:
# Preprocess: drop columns in training data that all values are the same
def find_constant_columns(df):
  """
  Finds columns in a Pandas DataFrame where all values are the same.

  Args:
      df: The Pandas DataFrame to analyze.

  Returns:
      A list of column names where all values are the same.
  """
  constant_cols = []
  for col in df.columns:
    if (df[col] == df[col].iloc[0]).all():
      constant_cols.append(col)
  return constant_cols
constant_columns = find_constant_columns(X)
print(constant_columns)


for col in constant_columns:
  X.drop(col, axis=1, inplace=True)


['Net_Income_Flag']


In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif



# 3. Remove highly correlated features
def remove_highly_correlated_features(df, threshold=0.9):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    df_reduced = df.drop(columns=to_drop)
    return df_reduced, to_drop

# 4. Remove low-variance features
def remove_low_variance_features(df, threshold=0.01):
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(df)
    selected_features = df.columns[selector.get_support()]
    df_reduced = df[selected_features]
    return df_reduced,selected_features, list(set(df.columns) - set(selected_features))

# 5. Select top features using supervised method (optional if you have target)
def select_top_features(X, y, k=50):
    selector = SelectKBest(score_func= f_classif, k=k)
    # selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X, y)
    selected_features = X.columns[selector.get_support()]
    reduced_features=list(set(X.columns) - set(selected_features))
    return X[selected_features], selected_features, reduced_features

# 6. Save final feature list
def save_feature_list(features, filepath):
    with open(filepath, 'w') as f:
        for feat in features:
            f.write(f"{feat}\n")

# 7. Full process
def feature_engineering_pipeline(df, save_featurelist_path=None, k_features=50):



    # Remove highly correlated features
    df, dropped_corr = remove_highly_correlated_features(df, threshold=0.9)
    print(f"X after removal: {len(df.columns)}.\nDropped {len(dropped_corr)} highly correlated features: {dropped_corr}.")

    # Remove low variance features
    # df,selected_features, dropped_variance = remove_low_variance_features(df, threshold=0.01)
    # print(f"X after removal: {len(df.columns)}.\nDropped {len(dropped_variance)} low-variance features: {dropped_variance}.")

    # Select top k features (supervised step)
    df, selected_features, reduced_features = select_top_features(df, y, k=k_features)
    print(f"X after removal: {len(df.columns)}.\nDropped {len(reduced_features)} features: {reduced_features}.")

    # Save the selected feature list if needed
    if save_featurelist_path:
        save_feature_list(selected_features, save_featurelist_path)
        print(f"Saved selected features to {save_featurelist_path}")

    return df, selected_features


X_selected, features = feature_engineering_pipeline(X, save_featurelist_path='selected_features_k_means.txt', k_features=50)


X after removal: 74.
Dropped 20 highly correlated features: ['ROA(A)_before_interest_and_%_after_tax', 'ROA(B)_before_interest_and_depreciation_after_tax', 'Realized_Sales_Gross_Margin', 'Pre-tax_net_Interest_Rate', 'After-tax_net_Interest_Rate', 'Continuous_interest_rate_(after_tax)', 'Net_Value_Per_Share_(A)', 'Net_Value_Per_Share_(C)', 'Per_Share_Net_profit_before_tax_(Yuan_¥)', 'Regular_Net_Profit_Growth_Rate', 'Net_worth/Assets', 'Operating_profit/Paid-in_capital', 'Net_profit_before_tax/Paid-in_capital', 'Current_Liabilities/Equity', 'Cash_Flow_to_Sales', 'Current_Liability_to_Liability', 'Current_Liability_to_Equity', 'Net_Income_to_Total_Assets', 'Gross_Profit_to_Sales', 'Liability_to_Equity'].
X after removal: 50.
Dropped 24 features: ['Average_Collection_Days', 'Working_capitcal_Turnover_Rate', 'Inventory_Turnover_Rate_(times)', 'Allocation_rate_per_person', 'Total_debt/Total_net_worth', 'Operating_Profit_Growth_Rate', 'Current_Ratio', 'Interest_Coverage_Ratio_(Interest_expen

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

# 3.2.1.  Cluster the training data from 3.1 into k-many subgroups

In [7]:
from sklearn.cluster import KMeans
import pandas as pd

# Let's say you already have X_train_scaled

# Apply KMeans
kmeans = KMeans(n_clusters=9, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels back to original dataframe
df['ClusterID'] = cluster_labels  # Add new column
df.to_csv('train_with_clusters.csv', index=False)
joblib.dump(kmeans, 'kmeans_model.pkl')

['kmeans_model.pkl']

# 3.2.2. Report the number of companies and the proportion of bankrupted companies in each subgroup.

In [8]:
import pandas as pd

# Load the train data that has 'ClusterID' and 'Bankrupt?' columns
df = pd.read_csv('train_with_clusters.csv')

# Group by ClusterID and check number of bankrupted companies in each cluster
cluster_summary = df.groupby('ClusterID')['Bankrupt?'].sum()

# Find clusters with 0 bankrupt companies
no_bankrupt_clusters = cluster_summary[cluster_summary == 0].index.tolist()

# Output
print(f"Clusters with NO bankrupt companies: {no_bankrupt_clusters}")
print(f"Total number of such clusters: {len(no_bankrupt_clusters)}")


Clusters with NO bankrupt companies: [1]
Total number of such clusters: 1


In [16]:

length_cluster=[len(df[df['ClusterID']==i]) for i in range(9)]


# Group by 'ClusterID' and count the number of companies with 'Bankrupt?' equal to 0
y_0_count = df.groupby('ClusterID')['Bankrupt?'].apply(lambda x: (x == 0).sum()).reset_index(name='Non-Bankrupt(y=0)')

# Merge the 'y_0_count' into the 'result' DataFrame
result = pd.merge(cluster_summary, y_0_count, on='ClusterID', how='left')

# Total samples in each cluster
length_cluster_series = pd.Series(length_cluster, name='total', index=cluster_summary.index)
result = pd.concat([result, length_cluster_series], axis=1)  # Concatenate along columns (axis=1)


# proportion of bankrupt companies in each cluster
result['Proportion'] = result['Bankrupt?'] / result['total']


result

,ClusterID,Bankrupt?,Non-Bankrupt(y=0),total,Proportion
0,0,29,1167,1196,0.024247
1,1,0,550,550,0.000000
2,2,82,1681,1763,0.046512
3,3,1,0,1,1.000000
4,4,2,0,2,1.000000
5,5,1,28,29,0.034483
6,6,72,262,334,0.215569
7,7,1,6,7,0.142857
8,8,10,1915,1925,0.005195


# 3.2.3. Identify unique or helpful characteristics in each subgroup.
Features that are important for one cluster may have following characteristics:

## 1. Unique from other clusters. This means the value of this feature is high in this cluster, but low in other cluster.

## 2. Common within same cluster. The values of this feature have low variance within this cluster.
## 3. Relevant to target 'bankrupt?'

In [ ]:
X_selected['ClusterID'] = cluster_labels

group_means = X_selected.groupby('ClusterID').mean()

# Standard deviation within each cluster for finding features that have smallest variance
cluster_std = X_selected.groupby('ClusterID').agg(['std'])
cluster_std.columns = cluster_std.columns.droplevel(1)

In [ ]:

# Calculate the average of all average values across all clusters
all_avg_means = group_means.mean()

# Calculate the absolute difference between each cluster's average and the overall average
# this can measure how far one feature is apart from other clusters.
diff_from_overall_avg = abs(group_means - all_avg_means)


In [ ]:
def get_important_features(cluster):
    # Select the 2 features with the largest differences for ClusterID 2
    cluster_2_farthest_features = diff_from_overall_avg.loc[cluster].sort_values(ascending=False).head(2).index.tolist()

    # Select 2 features with the lowest standard deviation for ClusterID 2
    cluster_2_lowest_std_features = cluster_std.loc[cluster].sort_values( ascending=True).head(2).index.tolist()

    important_features = set(cluster_2_farthest_features) | set(cluster_2_lowest_std_features)

    return important_features


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os



def visualize(cluster):
    print('=='*60)
    print(f"Visualizing cluster {cluster}:")
    important_features = get_important_features(cluster)

    # Plot bar plots of feature means per cluster using subplots
    num_features = len(important_features)
    num_cols = 4  # Number of columns in the subplot grid
    num_rows = (num_features + num_cols - 1) // num_cols  # Calculate number of rows



    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 3 * num_rows))  # Create subplots grid
    axes = axes.flatten()  # Flatten the axes array for easier indexing



    # Identify the most relevant to target 'bankrupt?'
    X_selected['Bankrupt?']=y
    df_cluster = X_selected[X_selected['ClusterID'] == cluster]

    related=pd.DataFrame()
    related['key']=features
    related_to_bankrupt=[]
    for i in range(len(features)):
        feature_mean = df_cluster.groupby('Bankrupt?')[features[i]].mean().to_list()
        if len(feature_mean) >= 2:
            related_to_bankrupt.append(abs(feature_mean[0]-feature_mean[1]))
        else:
            # Handle cases where feature_mean has less than 2 elements,
            # e.g., append 0 or some other default value
            related_to_bankrupt.append(0)  # Appending 0 as a default
    related['val']=related_to_bankrupt
    related_feature_sorted=related.sort_values(by='val',ascending=False)

    related_feature=related_feature_sorted.iloc[:4]['key'].to_list()

    for i, feature in enumerate(related_feature):
        feature_mean = df_cluster.groupby('Bankrupt?')[feature].mean()

        feature_mean.plot(kind='bar', ax=axes[i])
        axes[i].set_title(f'{feature}')
        axes[i].set_xlabel('Bankrupt?')
        axes[i].set_ylabel(feature)
    fig.suptitle('Most relevant Feature', fontsize=16)
    plt.tight_layout()
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 3 * num_rows))
    axes = axes.flatten()

    for i, feature in enumerate(important_features):

        mean_feature = df.groupby('ClusterID')[feature].mean()
        mean_feature.plot(kind='bar', ax=axes[i])  # Plot on the corresponding subplot
        axes[i].set_title(f'{feature}')
        axes[i].set_xlabel('ClusterID')
        axes[i].set_ylabel(feature)
    fig.suptitle('Average feature value per cluster', fontsize=16)
    plt.tight_layout()

    # Step 5: Plot boxplots for feature distributions across clusters using subplots
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 3 * num_rows))
    axes = axes.flatten()

    for i, feature in enumerate(important_features):
        sns.boxplot(x='ClusterID', y=feature, data=df, ax=axes[i])
        axes[i].set_title(f'{feature} ')
        axes[i].set_xlabel('ClusterID')
        axes[i].set_ylabel(feature)
    fig.suptitle('Feature Distribution per Cluster', fontsize=16)
    plt.tight_layout()
    plt.show()



In [ ]:

for i in range(9):
    visualize(i)

In [ ]:
import matplotlib.pyplot as plt

# Set larger figure size
plt.figure(figsize=(20, 15))  # Wider and taller

# Plot histograms
X_selected.hist(bins=30, figsize=(20, 15))  # Adjust number of bins

# Use tight layout to automatically fix overlapping titles and axes
plt.tight_layout()

# Optional: Adjust space manually if needed
plt.subplots_adjust(hspace=0.5, wspace=0.4)

# Show plot
plt.show()